# DuckDB로 277만 줄 다루기

pandas로 nationwide 277만 행을 열면 1GB 넘게 메모리를 먹고 오래 걸립니다.
DuckDB는 CSV를 직접 읽어 필요한 집계만 계산해서, 메모리를 거의 쓰지 않고 끝냅니다.
이 노트북은 [상권분석 대시보드](https://github.com/team-datapopcorn/sanggwon-dashboard)의
데이터 파이프라인에서 쓰는 핵심 두 가지 방법을 분리해 연습해 봅니다.

1. 파일을 통째로 올리지 않고 SQL로 집계하기
2. 브라우저에서 쓸 수 있게 결과를 미리 압축하기

In [ ]:
!pip -q install duckdb pandas

## 0. 예제 파일 준비

수업에서는 작은 CSV를 직접 만들어서 씁니다.
실제 상권 데이터를 쓰려면 [data.go.kr](https://www.data.go.kr/data/15083033/fileData.do)에서
ZIP을 받아 압축을 푼 뒤 아래 `csv_dir`만 바꾸면 됩니다.

In [ ]:
import os, random, pathlib
import pandas as pd

random.seed(42)
csv_dir = pathlib.Path("/content/sample_data")
csv_dir.mkdir(parents=True, exist_ok=True)

COLUMNS = ["상호명", "상권업종대분류명", "상권업종소분류명", "시도명", "시군구명", "경도", "위도"]
SIDO = ["서울특별시", "경기도", "부산광역시", "대구광역시"]
SGG = {"서울특별시": ["강남구", "마포구", "종로구"],
       "경기도": ["성남시 분당구", "용인시 기흥구"],
       "부산광역시": ["해운대구", "중구"],
       "대구광역시": ["수성구", "달서구"]}
MAJOR = ["음식", "소매", "수리·개인", "과학·기술", "교육", "부동산", "숙박"]
SMALL = ["카페", "편의점", "치킨", "미용실", "백반/한정식", "부동산 중개/대리업",
         "입시·교과학원", "슈퍼마켓"]

rows = []
for sido in SIDO:
    for sgg in SGG[sido]:
        for _ in range(1500):
            rows.append({
                "상호명": random.choice(["아지트", "스타벅스", "파리바게뜨"]),
                "상권업종대분류명": random.choice(MAJOR),
                "상권업종소분류명": random.choice(SMALL),
                "시도명": sido,
                "시군구명": sgg,
                "경도": random.uniform(126.0, 129.5),
                "위도": random.uniform(35.0, 38.0),
            })

df = pd.DataFrame(rows)[COLUMNS]
out = csv_dir / "상가_예제.csv"
df.to_csv(out, index=False, encoding="utf-8")
print(f"{out} : {len(df):,}행")

## 1. read_csv로 폴더째 읽기

파일을 여러 개로 나누어 받아도, glob 한 줄이면 하나의 테이블처럼 읽힙니다.
따로 합치거나(index로 이어붙이기) 메모리에 다 올릴 필요가 없습니다.

In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"""
    create table shop as
    select * from read_csv('{csv_dir}/*.csv', header=true, union_by_name=true)
""")

print(con.execute("select count(*) from shop").fetchone()[0], "행")

In [ ]:
# group by는 DuckDB가 처리한다. pandas로 옮기지 않는다.
top = con.execute("""
    select 시도명, 시군구명, count(*) as 업소수
    from shop
    group by 1, 2
    order by 3 desc
    limit 10
""").df()   # 마지막에만 pandas로 넘긴다
top

### 연습 1

1. 카페와 편의점 수를 시군구별로 구하기
2. 각 시군구에서 가장 업소가 많은 소분류 하나만 뽑기
3. 두 시군구(예: 강남구, 중구)를 골라 업종 구성을 나란히 비교하기

힌트: `where 상권업종소분류명 in ('카페','편의점')`, `qualify row_number() over (...)`

In [ ]:
con.execute("""
    select 시군구명,
           count(*) filter (where 상권업종소분류명='카페') as 카페,
           count(*) filter (where 상권업종소분류명='편의점') as 편의점
    from shop
    group by 1
    order by 2 desc
""").df()

## 2. 반경 안에 뭐가 있는지 세기

위경도가 있으므로 거리 계산이 가능합니다. 하버드 공식으로 두 점 사이의 great-circle
거리(미터)를 구합니다.

In [ ]:
# 강남역(37.5008, 127.0364) 반경 3km
LAT, LON, R = 37.5008, 127.0364, 3000

nearby = con.execute(f"""
    select 상권업종대분류명, count(*) as 점포수,
           round(100.0 * count(*) / sum(count(*)) over (), 1) as 비중
    from shop
    where 6371000 * 2 * asin(sqrt(
            power(sin(radians((위도 - {LAT}) / 2)), 2)
            + cos(radians({LAT})) * cos(radians(위도))
            * power(sin(radians((경도 - {LON}) / 2)), 2)
    )) <= {R}
    group by 1
    order by 2 desc
""").df()
nearby

### 연습 2

반경을 1km, 3km, 5km으로 바꿔 가며 카페 비중이 어떻게 달라지는지 보세요.
가까운 거리일수록 카페가 늘면 "상권이 성숙했다"는 신호로 읽을 수 있습니다.
단, 이게 실제 매출이나 손익을 뜻하지는 않습니다.

## 3. 지도를 그릴 수 있게 압축하기

원본 점을 그대로 JSON으로 저장하면 수백 MB가 됩니다.
지도를 그릴 때는 좌표 자체가 아니라 **격자별 개수**만 있으면 충분합니다.

0.02도 격자(위도 기준 약 2.2km)로 묶으면 점이 크게 줄고,
어떤 지역이 붐빴는지, 어떤 지역이 비었는지가 오히려 선명해집니다.

In [ ]:
grid = con.execute("""
    select floor(위도 / 0.02) * 0.02 as lat,
           floor(경도 / 0.02) * 0.02 as lon,
           count(*) as n,
           count(*) filter (where 상권업종소분류명='카페') as 카페
    from shop
    group by 1, 2
    order by 3 desc
""").df()

print(f"원본 {len(df):,}점 -> 격자 {len(grid):,}셀")
grid.head()

In [ ]:
# 압축률 확인
raw_size = df.to_json(orient='records').encode('utf-8')
grid_size = grid.to_json(orient='records').encode('utf-8')
print(f"원본 JSON {len(raw_size)/1024:.0f}KB")
print(f"격자 JSON {len(grid_size)/1024:.0f}KB")
print(f"{len(grid)/len(df)*100:.1f}% 의 크기")

## 4. JSON으로 저장해서 브라우저에서 쓰기

이렇게 만든 JSON을 GitHub Pages 같은 정적 호스팅에 올리면
서버 없이 브라우저에서 바로 표와 지도를 그릴 수 있습니다.

실제 대시보드: `scripts/build_aggregates.py` 참고

## 정리

- **CSV를 통째로 메모리에 올리지 않는다.** `read_csv`에 glob을 주고, 집계는 DB가 하게 둡니다.
- **pandas로 옮기는 건 마지막 결과만.** `.df()`를 셀 끝에 한 번만 붙입니다.
- **지도용으로는 점을 격자로 압축합니다.** 정보가 줄지 않고 용량이 크게 줄어듭니다.

다음 단계로 넘어가기 전에, 이 데이터로 **구할 수 없는 것**을 한 번 생각해 보세요.
폐업률, 매출, 실제 체류 인구. 이 저장소의 대시보드는 못 하는 질문을 UI에도 그대로 적어뒀습니다.

In [ ]:
print("끝. 전체 저장소: https://github.com/team-datapopcorn/sanggwon-dashboard")